In [0]:
dbutils.widgets.text("order_status", "")
dbutils.widgets.text("order_status_group", "")

order_status = dbutils.widgets.get("order_status").strip().upper()
order_status_group = dbutils.widgets.get("order_status_group").strip()

qry = f'''
SELECT
  order_status,
  order_status_group,
  COUNT(*) AS status_count
FROM ORDERS_SALES.SILVER.orders_enriched
WHERE upper(order_status) = '{order_status}'
AND order_status IS NOT NULL
GROUP BY order_status, order_status_group
ORDER BY status_count DESC
'''

result_df = spark.sql(qry)

rows = result_df.collect()

combination_array = [
        {
            "status_code" : row["status_code"],
            "order_status_group" : row["order_status_group"]
        }
        for row in rows
]

array_length = len(combination_array)

print('Generated Combination:')
print(combination_array)
print(f'Total Combination: {array_length}')

dbutils.jobs.taskValues.set(key="combination_array", value=combination_array)
dbutils.jobs.taskValues.set(key="array_length", value=array_length)